# Compare $\alpha_{base}$ vs $\alpha_{cat1}$

Loads `combined_alpha_sep.csv` (output of `combine_alpha_sep.py`), applies quality cuts, and makes a scatter plot of the baseband spectral index vs the CHIME/CAT1 spectral index with error bars.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## Settings

Edit these as needed -- input path and selection criteria.

In [ ]:
COMBINED_CSV = "/home/thsu/FRB_population/CHIME_spectra/combined_alpha_sep.csv"

# ---- selection criteria ----
DM_MAX = 500                 # keep rows with dm < DM_MAX
MIN_BANDWIDTH_BASE_MHZ = 200 # keep rows with (freq_top_base - freq_low_base) > this, in MHz

# set to True to also require finite/non-null alpha + error values on both sides
REQUIRE_FINITE_ALPHA = True

## Load data

In [ ]:
df = pd.read_csv(COMBINED_CSV)
print(f"loaded {len(df)} rows")
df.head()

## Apply criteria

Note: `freq_top_base`/`freq_low_base` are assumed to be in MHz already. If your columns are in GHz, either convert here or change `MIN_BANDWIDTH_BASE_MHZ` to match your units.

In [ ]:
mask = pd.Series(True, index=df.index)

mask &= df["dm"] < DM_MAX
mask &= (df["freq_top_base"] - df["freq_low_base"]) > MIN_BANDWIDTH_BASE_MHZ

if REQUIRE_FINITE_ALPHA:
    for col in ["alpha_base", "alpha_err_base", "alpha_cat1", "alpha_err_cat1"]:
        mask &= df[col].notna()

sel = df[mask].copy()
print(f"{len(sel)} / {len(df)} rows pass the criteria")
print(f"  DM < {DM_MAX}")
print(f"  (freq_top_base - freq_low_base) > {MIN_BANDWIDTH_BASE_MHZ} MHz")
sel.head()

## Scatter plot with error bars (filtered by selection criteria)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.errorbar(
    sel["alpha_base"],
    sel["alpha_cat1"],
    xerr=sel["alpha_err_base"],
    yerr=sel["alpha_err_cat1"],
    fmt="o",
    markersize=5,
    ecolor="gray",
    elinewidth=1,
    capsize=2,
    alpha=0.8,
)

# 1:1 reference line
lo = min(sel["alpha_base"].min(), sel["alpha_cat1"].min())
hi = max(sel["alpha_base"].max(), sel["alpha_cat1"].max())
pad = 0.05 * (hi - lo)
ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", linewidth=1, label="1:1")

ax.set_xlim(lo - pad, hi + pad)
ax.set_ylim(lo - pad, hi + pad)
ax.set_xlabel(r"$\alpha_{base}$")
ax.set_ylabel(r"$\alpha_{cat1}$")
ax.set_title(
    f"Baseband vs CAT1 spectral index\n"
    f"(DM < {DM_MAX}, bandwidth$_{{base}}$ > {MIN_BANDWIDTH_BASE_MHZ} MHz, N={len(sel)})"
)
ax.legend()
ax.set_aspect("equal", adjustable="box")
fig.tight_layout()

fig.savefig("alpha_base_vs_cat1.png", dpi=200)
plt.show()

## Interactive scatter plot: click a point to identify its FRB

This needs an interactive matplotlib backend, not the static inline one. In Jupyter/VS Code:
```
pip install ipympl
```
then run `%matplotlib widget` (below) *before* plotting. If `ipympl` isn't available, `%matplotlib notebook` also works in classic Jupyter. Click near a point and its `frb_name`, alpha values, and errors are printed below the plot and shown as an on-plot label.

In [ ]:
%matplotlib widget

In [ ]:
fig2, ax2 = plt.subplots(figsize=(7, 7))

ax2.errorbar(
    sel["alpha_base"],
    sel["alpha_cat1"],
    xerr=sel["alpha_err_base"],
    yerr=sel["alpha_err_cat1"],
    fmt="o",
    markersize=5,
    ecolor="gray",
    elinewidth=1,
    capsize=2,
    alpha=0.8,
    zorder=2,
)

# invisible scatter laid on top, purely so we have clickable ('pickable') points
# (errorbar containers aren't easily pickable themselves)
sc = ax2.scatter(
    sel["alpha_base"], sel["alpha_cat1"],
    s=120, alpha=0, picker=True, zorder=3,
)

lo2 = min(sel["alpha_base"].min(), sel["alpha_cat1"].min())
hi2 = max(sel["alpha_base"].max(), sel["alpha_cat1"].max())
pad2 = 0.05 * (hi2 - lo2)
ax2.plot([lo2 - pad2, hi2 + pad2], [lo2 - pad2, hi2 + pad2], "k--", linewidth=1, label="1:1")
ax2.set_xlim(lo2 - pad2, hi2 + pad2)
ax2.set_ylim(lo2 - pad2, hi2 + pad2)
ax2.set_xlabel(r"$\alpha_{base}$")
ax2.set_ylabel(r"$\alpha_{cat1}$")
ax2.set_title("Click a point to identify its FRB (filtered data)")
ax2.legend()
ax2.set_aspect("equal", adjustable="box")

frb_names = sel["frb_name"].to_numpy()
alpha_base_vals = sel["alpha_base"].to_numpy()
alpha_cat1_vals = sel["alpha_cat1"].to_numpy()

annot = ax2.annotate(
    "", xy=(0, 0), xytext=(15, 15), textcoords="offset points",
    bbox=dict(boxstyle="round", fc="white", ec="gray", alpha=0.95),
    arrowprops=dict(arrowstyle="->"),
)
annot.set_visible(False)

def onpick(event):
    if event.artist != sc:
        return
    idx = event.ind[0]  # closest point if several are within the pick radius
    name = frb_names[idx]
    xb, yc = alpha_base_vals[idx], alpha_cat1_vals[idx]
    annot.xy = (xb, yc)
    annot.set_text(f"{name}\nalpha_base={xb:.3f}\nalpha_cat1={yc:.3f}")
    annot.set_visible(True)
    fig2.canvas.draw_idle()
    print(f"FRB {name}: alpha_base={xb:.3f}, alpha_cat1={yc:.3f}")

fig2.canvas.mpl_connect("pick_event", onpick)
plt.show()

Switch back to the static/inline backend so all plots below this are normal (non-interactive) images, not the widget's zoomable canvas with a live x/y cursor readout.

In [ ]:
%matplotlib inline

## Peak frequency histograms

Histograms of `peak_freq_cat1` and `peak_freq_gauss_base` using **all data** (not the DM/bandwidth-filtered `sel`), plus the ratio of counts in the 700-800 MHz band to the 400-500 MHz band for each.

In [ ]:
PEAK_FREQ_COLS = ["peak_freq_cat1", "peak_freq_gauss_base"]
BAND_HIGH = (700, 800)  # MHz
BAND_LOW = (400, 500)   # MHz
BINS = np.arange(400, 810, 10)  # 10 MHz bins spanning both bands; widen/narrow as needed

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

band_counts = {}
for ax, col in zip(axes, PEAK_FREQ_COLS):
    vals = df[col].dropna()
    ax.hist(vals, bins=BINS, color="steelblue", edgecolor="black", alpha=0.8)
    ax.axvspan(*BAND_LOW, color="tab:orange", alpha=0.15, label=f"{BAND_LOW[0]}-{BAND_LOW[1]} MHz")
    ax.axvspan(*BAND_HIGH, color="tab:green", alpha=0.15, label=f"{BAND_HIGH[0]}-{BAND_HIGH[1]} MHz")
    ax.set_xlabel(f"{col} (MHz)")
    ax.set_ylabel("count")
    ax.set_title(col)
    ax.legend(fontsize=8)

    n_high = vals.between(*BAND_HIGH).sum()
    n_low = vals.between(*BAND_LOW).sum()
    ratio = n_high / n_low if n_low > 0 else np.nan
    band_counts[col] = {"n_700_800": n_high, "n_400_500": n_low, "ratio": ratio}

    ax.text(
        0.97, 0.95,
        f"N(700-800)={n_high}\nN(400-500)={n_low}\nratio={ratio:.3f}",
        transform=ax.transAxes,
        ha="right", va="top",
        fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="gray", alpha=0.9),
    )

fig.tight_layout()
fig.savefig("peak_freq_histograms.png", dpi=200)
plt.show()

for col, d in band_counts.items():
    print(
        f"{col}: N(700-800 MHz)={d['n_700_800']}, N(400-500 MHz)={d['n_400_500']}, "
        f"ratio(700-800 / 400-500)={d['ratio']:.3f}"
    )

## Peak frequency (Gaussian fit): baseband vs CAT1, all FRBs

Scatter plot of `peak_freq_gauss_base` vs `peak_freq_gauss_cat1` with error bars from `peak_freq_gauss_err_base` / `peak_freq_gauss_err_cat1`, using all data (no DM/bandwidth cuts).

In [ ]:
peak_df = df.dropna(
    subset=[
        "peak_freq_gauss_base", "peak_freq_gauss_err_base",
        "peak_freq_gauss_cat1", "peak_freq_gauss_err_cat1",
    ]
)

fig3, ax3 = plt.subplots(figsize=(7, 7))

ax3.errorbar(
    peak_df["peak_freq_gauss_base"],
    peak_df["peak_freq_gauss_cat1"],
    xerr=peak_df["peak_freq_gauss_err_base"],
    yerr=peak_df["peak_freq_gauss_err_cat1"],
    fmt="o",
    markersize=5,
    ecolor="gray",
    elinewidth=1,
    capsize=2,
    alpha=0.8,
)

# 1:1 reference line
lo3 = min(peak_df["peak_freq_gauss_base"].min(), peak_df["peak_freq_gauss_cat1"].min())
hi3 = max(peak_df["peak_freq_gauss_base"].max(), peak_df["peak_freq_gauss_cat1"].max())
pad3 = 0.05 * (hi3 - lo3)
ax3.plot([lo3 - pad3, hi3 + pad3], [lo3 - pad3, hi3 + pad3], "k--", linewidth=1, label="1:1")

ax3.set_xlim(lo3 - pad3, hi3 + pad3)
ax3.set_ylim(lo3 - pad3, hi3 + pad3)
ax3.set_xlabel("peak_freq_gauss_base (MHz)")
ax3.set_ylabel("peak_freq_gauss_cat1 (MHz)")
ax3.set_title(f"Gaussian peak frequency: baseband vs CAT1 (all data, N={len(peak_df)})")
ax3.legend()
ax3.set_aspect("equal", adjustable="box")
fig3.tight_layout()

fig3.savefig("peak_freq_gauss_base_vs_cat1.png", dpi=200)
plt.show()

## Peak frequency error histograms

Histograms of `peak_freq_gauss_err_base` and `peak_freq_gauss_err_cat1`, using all data.

In [ ]:
ERR_COLS = ["peak_freq_gauss_err_base", "peak_freq_gauss_err_cat1"]

fig4, axes4 = plt.subplots(1, 2, figsize=(12, 5))

for ax, col in zip(axes4, ERR_COLS):
    vals = df[col].dropna()
    ax.hist(vals, bins=30, color="steelblue", edgecolor="black", alpha=0.8)
    ax.set_xlabel(f"{col} (MHz)")
    ax.set_ylabel("count")
    ax.set_title(f"{col} (N={len(vals)}, median={vals.median():.1f} MHz)")

fig4.tight_layout()
fig4.savefig("peak_freq_gauss_err_histograms.png", dpi=200)
plt.show()